方案一：Tesseract + pytesseract（最经典，适合新手入门）

In [5]:
import pytesseract
from PIL import Image

# Windows用户可能需要指定Tesseract的安装路径，取消下一行的注释并修改为你的路径
pytesseract.pytesseract.tesseract_cmd = r'D:\Program Files\Tesseract-OCR\tesseract.exe'

# 打开图片文件
image = Image.open('../img/img.png')

# 识别图片中的文字，lang='chi_sim' 表示使用简体中文，'eng'表示英文
# 可以用 '+' 同时识别多种语言，如 lang='chi_sim+eng'
text = pytesseract.image_to_string(image, lang='chi_sim')

# 打印识别出的文字
print(text)

RCSDN 的 风 格 设 计 , 包 合 以 下 特 点 :
千 醒 目 的 沧 颜 和 分 殷

2 屹 4 代 矿 片 段

3 切 比 表 恒 叶 定 战 坂 暨

4 箩 见 问 颂
5 项 目 源 围 涛 引
6 . 扩 展 学 习 资 源

OCR 技 札 团 ! 如 有 羰 闯 次 沃 地 # 区 留 言 讨 论 , 点 击 关 注 获 取 曰 多 Pyihon 吴 战 芬 !



In [7]:

import os
print("当前工作目录:", os.getcwd())
print("图片路径:", os.path.join(os.path.dirname(os.path.abspath('__file__')), 'img', 'img.png'))
print("文件是否存在:", os.path.exists(os.path.join(os.path.dirname(os.path.abspath('__file__')), 'img', 'img.png')))

当前工作目录: D:\project\python\aidemo
图片路径: D:\project\python\aidemo\img\img.png
文件是否存在: True


# 方案二：EasyOCR（上手最快，深度学习驱动）
如果你想获得开箱即用的体验，且希望代码极简，EasyOCR是不错的选择。它基于深度学习，对倾斜、模糊文字的容忍度更高。

In [2]:

# ... existing code ...
import easyocr_demo

# 初始化读取器，只需指定语言
# 这步会下载模型（约500MB），请耐心等待
reader = easyocr.Reader(['ch_sim', 'en'])

# 执行识别，返回 (边界框, 文字, 置信度)
result = reader.readtext('./img/img.png')

# 提取并打印识别出的文字
for (bbox, text, prob) in result:
    print(f"文字: {text}, 置信度: {prob:.2f}")
# ... existing code ...

Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Administrator\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Administrator\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Administrator\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Administrator\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('PP-OCRv5_server_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manual

TypeError: PaddleOCR.predict() got an unexpected keyword argument 'cls'

# 方案三：PaddleOCR（中文识别最强，适合企业级应用）
如果你的场景是中文文档，且追求极致准确率（特别是发票、合同等），推荐百度的PaddleOCR。它针对中文进行了专门优化，准确率领先。

In [7]:

# ... existing code ...
from paddleocr import PaddleOCR

# 全局只初始化一次
if 'ocr' not in globals():
    ocr = PaddleOCR(use_angle_cls=True, lang='ch')

def recognize_image(image_path):
    """识别图片中的文字"""
    # 使用新的 predict 方法替代已弃用的 ocr 方法
    result = ocr.predict(image_path)

    texts = []

    # 打印原始结果结构,方便调试
    print("原始结果类型:", type(result))
    print("原始结果:", result)
    print("="*50)

    # 根据实际返回格式解析
    if isinstance(result, list):
        for idx, line in enumerate(result):
            print(f"第 {idx} 个元素类型: {type(line)}")

            if line is None:
                continue

            # 如果 line 是字典
            if isinstance(line, dict):
                print(f"字典键: {line.keys()}")
                # 尝试从字典中提取文本信息
                if 'rec_texts' in line:
                    rec_texts = line['rec_texts']
                    rec_scores = line.get('rec_scores', [])

                    for i, text in enumerate(rec_texts):
                        confidence = rec_scores[i] if i < len(rec_scores) else 0
                        texts.append({
                            'text': text,
                            'confidence': confidence
                        })

            # 如果 line 是列表
            elif isinstance(line, list):
                print(f"列表长度: {len(line)}, 第一个元素: {line[0] if line else 'empty'}")

                for word_info in line:
                    try:
                        if isinstance(word_info, list) and len(word_info) >= 2:
                            bbox = word_info[0]
                            text_confidence = word_info[1]

                            if isinstance(text_confidence, tuple) and len(text_confidence) >= 2:
                                text = text_confidence[0]
                                confidence = text_confidence[1]

                                texts.append({
                                    'text': text,
                                    'confidence': confidence
                                })
                    except Exception as e:
                        print(f"处理出错: {e}, word_info: {word_info}")

    return texts

# 使用
results = recognize_image('../img/img.png')
print("\n=== 识别结果 ===")
for item in results:
    print(f"识别文本: {item['text']}, 置信度: {item['confidence']:.2f}")

原始结果类型: <class 'list'>
原始结果: [{'input_path': './img/img.png', 'page_index': None, 'doc_preprocessor_res': {'input_path': None, 'page_index': None, 'input_img': array([[[255, ..., 255],
        ...,
        [255, ..., 255]],

       ...,

       [[255, ..., 255],
        ...,
        [255, ..., 255]]], shape=(392, 889, 3), dtype=uint8), 'model_settings': {'use_doc_orientation_classify': True, 'use_doc_unwarping': True}, 'angle': 0, 'rot_img': array([[[255, ..., 255],
        ...,
        [255, ..., 255]],

       ...,

       [[255, ..., 255],
        ...,
        [255, ..., 255]]], shape=(392, 889, 3), dtype=uint8), 'output_img': array([[[254, ..., 254],
        ...,
        [255, ..., 255]],

       ...,

       [[255, ..., 255],
        ...,
        [255, ..., 255]]], shape=(392, 889, 3), dtype=uint8)}, 'dt_polys': [array([[ 0, 28],
       ...,
       [ 0, 49]], shape=(4, 2), dtype=int16), array([[  0, 103],
       ...,
       [  0, 124]], shape=(4, 2), dtype=int16), array([[  4, 149